In [1]:
from __future__ import annotations as _annotations

import matplotlib.pyplot as plt
import os

from dotenv import load_dotenv

load_dotenv()
from openai import OpenAI
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
client = OpenAI()

import numpy as np
from pathlib import Path
import sys

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import math
import random

from batch_tools.batch_manager import BatchManager
from models.address import Address, AddressType, AddressResult, SourceEnum

from typing import Any

import pandas as pd

import re
import json
from pydantic import ValidationError

from rapidfuzz import process, fuzz




In [99]:
def row_to_address(row):
    return Address(
        osm_id=row.get("osm_id"),
        city=row.get("city"),
        postcode=row.get("postcode"),
        street=row.get("street"),
        housenumber=row.get("housenumber"),
        lat=row.get("lat"),
        lon=row.get("lon"),
        type=AddressType.GOLD,
        source=SourceEnum.ADDRESS_DATABASE,
    )


# Funktion von ChatGPT
def parse_agent_response(response_text):
    """
    Code snippet by chat gpt!

    Prompt:
    Gebe mir eine Funktion, die den Output-Text in ein ordentlichen json verwandelt!
    """

    """
    Nimmt den Text-Output eines einzelnen Agenten/LLM-Calls
    und gibt ein Python-dict oder eine list zurück.
    """

    if response_text is None:
        return None

    # Falls schon dict/list übergeben wurde
    if isinstance(response_text, (dict, list)):
        return response_text

    text = str(response_text).strip()

    # Markdown-Codeblock entfernen, z.B. ```json ... ```
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text)

    # Direktes JSON parsen
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Falls vor/nach dem JSON noch Text steht:
    json_match = re.search(r"(\{.*\}|\[.*\])", text, flags=re.DOTALL)

    if json_match:
        json_text = json_match.group(1)
        try:
            return json.loads(json_text)
        except json.JSONDecodeError as e:
            raise ValueError(f"JSON gefunden, aber nicht parsebar: {e}\n\nGefundener Text:\n{json_text}")

    raise ValueError(f"Kein gültiges JSON im Agenten-Output gefunden:\n{text}")





def blur_gps(lat, lon, min_km=50, max_km=100):
    """
    Code snippet by chat gpt!

    Prompt:
    Wie kann ich gps-Koordinaten in python verwischen, so dass sie in einer random Richtung 50-100 km entfernt liegen?
    """


    """
    Verschiebt GPS-Koordinaten in eine zufällige Richtung
    um eine zufällige Distanz zwischen min_km und max_km.
    """

    # Erdradius in km
    R = 6371.0

    # Zufällige Distanz und Richtung
    distance = random.uniform(min_km, max_km)
    bearing = random.uniform(0, 2 * math.pi)

    # Eingabe in Radiant umwandeln
    lat1 = math.radians(lat)
    lon1 = math.radians(lon)

    # Distanz relativ zum Erdradius
    d = distance / R

    # Neue Koordinaten berechnen
    lat2 = math.asin(
        math.sin(lat1) * math.cos(d)
        + math.cos(lat1) * math.sin(d) * math.cos(bearing)
    )

    lon2 = lon1 + math.atan2(
        math.sin(bearing) * math.sin(d) * math.cos(lat1),
        math.cos(d) - math.sin(lat1) * math.sin(lat2)
    )

    # Zurück in Grad
    new_lat = math.degrees(lat2)
    new_lon = math.degrees(lon2)

    return new_lat, new_lon

def get_accuracy():
    i = random.randint(1,100)
    if i <= 35: return "high accuracy"
    if i <= 70: return "medium accuracy"
    if i <= 90: return "low accuracy"
    return "very low accuracy"

def new_gps(address, onscene):


    if onscene:
        acc = get_accuracy()

        if acc == "high accuracy": lat_n, lon_n = blur_gps(address.lat, address.lon, 0, 0.030)
        elif acc == "medium accuracy": lat_n, lon_n = blur_gps(address.lat, address.lon, 0, 0.050)
        elif acc == "low accuracy": lat_n, lon_n = blur_gps(address.lat, address.lon, 0, 0.100)
        else: lat_n, lon_n = blur_gps(address.lat, address.lon, 0, 0.500)

    else:
        acc = get_accuracy()
        lat_n, lon_n = blur_gps(address.lat, address.lon, 0.100, 100)

    return lat_n, lon_n, acc


def make_transcript(text):
    # Fakt die Probleme der Transkription

    SYSTEM_PROMPT_Transcript ="""
Du erhälts ein Gespräch zwischen Calltaker und Anrufer in einer Leitstelle.
Deine Aufgabe ist es, dass Gespräch so zu verändern, als wären das Gespräch aus einer Audio-Aufnahme Transkribiert worden.
Verändere nicht die Struktur des Dialogs oder den Inhalt.
Insbesondere solltest du aber die Begriffe verändern, die ein Transkriptionsmodell normalerweise nicht kennt, also Eigennamen, Straßen etc. Auch die Nummern solltest du ausschreiben.
Gib wieder ein json zurück, das genau die Form hat wie das ursprüngliche Gespräch
Gebe nur das JSON zurück"""

    user_prompt =f"""
    Das Gespräch: \n
    {text}
    Denk dran die Eigennamen zu verändern, so dass sie phonetisch ähnlich sind.
    """


    response = client.responses.create(
    model="gpt-5.4",
    instructions=SYSTEM_PROMPT_Transcript,
    input=user_prompt
)

    return parse_agent_response(response.output_text)

def get_level_text(level):
    if level == 0:
        return "Die Antworten des Anrufers sollen natürlich, teilweise unsicher und laienhaft klingen. Der Anrufer weiß aber wo er ist"
    if level == 1:
        return "Der Anrufer ist aufgeregt und antwortet nicht direkt auf alle Fragen. Er lässt sich aber beruhigen und gut durch Gespräch führen."
    if level == 2:
        return "Der Anrufer ist sehr aufgeregt und antwortet nicht direkt auf die Fragen. Er versucht zunächst zu erzählen was passiert ist. Erst nach genauer Frage nach der Örtlichkeit wird er darauf antworten, die Kommunikation ist deutlich herausfordernder."
    if level == 3:
        return "Der Anrufer ist extrem aufgeregt und versteht den Calltaker sehr schlecht. Es gibt deutliche Kommunikationsprobleme, die Fragen nach der Adresse sind extrem erschwert."

def get_hints_text(hints):
    if hints == 0:
        return "Der Anrufer ist selbst vor Ort und ist ortskundig. Die Adresse ist ihm gut bekannt."
    if hints == 1:
        return "Der Anrufer ist selbst vor Ort, ist aber etwas unsicher wo er sich befindet. Details wie die Hausnummer müssen erst erfragt werden"
    if hints == 2:
        return "Der Anrufer ist vor Ort aber ist nicht ortskundig. Er weiß nicht die genaue Straßen- oder Ortsbezeichnung, erst nach expliziter nachfrage."
    if hints == 3:
        return "Der Anrufer ist selbst nicht vor Ort, sondern ruft für eine dritte Person an. Allerdings ist die Person ortskundig und kennt die Adressdetails."
    if hints == 4:
        return "Der Der Anrufer ist selbst nicht vor Ort, sondern ruft für eine dritte Person an. Der Anrufer ist selbst nicht so gut ortskundig und kennt die genauen Adressdetails nur auf genaue Nachfrage."

def get_user_prompt_case(row, level, hints):


    level_text = get_level_text(level)
    hints_text = get_hints_text(hints)


    return  f"""
    Erzeuge den Anfang eines Notrufdialogs.

    Adresse:
    Straße: {row.get("street")}
    Hausnummer: {row.get("housenumber")}
    Ort: {row.get("city")}

    Schwierigkeit: {level_text}
    Hinweise: {hints_text}
    """

def get_system_prompt_case(level, hints):

    level_text = get_level_text(level)
    hints_text = get_hints_text(hints)

    return f"""
    Du erzeugst synthetische deutsche Notrufdialoge für Forschungs- und Testzwecke.

    Rolle:
    - Der Dialog ist zwischen einer Leitstellenperson und einer anrufenden Person.
    - Die anrufende Person beschreibt einen Notfall und den Einsatzort.

    Wichtig:
    - Nutze exakt die übergebene Adresse als Einsatzort.
    - Die Adresse soll realistisch in den Dialog eingebaut werden.
    - Die Leitstellenperson fragt nach Ort und Situation. Konzentriere dich erst auf die Abfrage des Einsatortes mache danach eine kurze Abfrage zur Situation.

    Deine Abfrage als Calltaker beginnt mit "Notruf Feuerwehr und Rettungsdienst, in welchem Ort ist der Notfall?"

    Die Schwierigkeit der Kommunikation zwischen Anrufer und Calltaker lässt sich so beschreiben: {level_text}
    Weitere Hinweise auf den Call die beachtet werden sollen: {hints_text}


    Ausgabeformat:

    {{  "dialogue": [
        {{"speaker": "calltaker", "text": "..."}},
        {{"speaker": "caller", "text": "..."}}
      ]
    }}

    """


def get_case(model="gpt-5.4"):
    # Erzeugt einen Case mit allen notwendigen Infos: Dialog, Trankription, Adresse, AML-Daten
    address = addresses.sample()
    level = random.randint(0, 3)
    hints = random.randint(0, 4)
    add = row_to_address(address.iloc[0])

    user_prompt = get_user_prompt_case(address, level, hints)
    system_prompt = get_system_prompt_case(level, hints)

    response = client.responses.create(
        model=model,
        instructions=system_prompt,
        input=user_prompt
    )


    if hints <3 :
        onscene = True
    else: onscene = False

    dialogue = parse_agent_response(response.output_text)

    aml_lat, aml_lon, aml_acc = new_gps(add, onscene)

    return {'gold_address':add, 'onscene':onscene, 'dialogue': dialogue, 'dialogue_transcripted': make_transcript(dialogue), 'aml_lat': aml_lat, 'aml_lon': aml_lon, 'aml_acc': aml_acc}

#Address Agent
def haversine_km(lat1, lon1, lat2, lon2):
    lat1 = math.radians(lat1)
    lon1 = math.radians(lon1)
    lat2 = math.radians(lat2)
    lon2 = math.radians(lon2)
    R = 6371.0088
    dlat, dlon = lat2 - lat1, lon2 - lon1
    h = math.sin(dlat/2)**2 + math.cos(lat1)*math.cos(lat2) * math.sin(dlon/2)**2
    return 2 * R * math.asin(math.sqrt(h))

def llm_to_addresses(
    llm_output: dict[str, Any] | None
) -> list[Address]:
    """
    Wandelt die LLM-Ausgabe in eine Liste von Address-Objekten um und setzt die Quelle auf TRANSCRIPT.
    """

    if not llm_output:
        return []

    try:
        result = AddressResult.model_validate(llm_output)
    except ValidationError as exc:
        print(f"Adressausgabe konnte nicht validiert werden: {exc}")
        return []

    return [
        address.model_copy(
            update={"source": SourceEnum.TRANSCRIPT}
        )
        for address in result.addresses
    ]


def address_agent(dialogue):

    SYSTEM_PROMPT = """
    Du erhälts das Transkript eines deutschen Notrufgesprächs. Filtere alle Informationen zur Adressangabe aus dem Notrufgespräch. Da es ein Transkript ist, können Transkriptionsfehler entstanden sein, achte also auf phonetische Ähnlichkeiten.
     Gebe nur ein JSON zurück, das folgende Form hat:
     {
      "addresses": [
        {
          "city": "string | null",
          "street": "string | null",
          "house_number": "string | null",
          "postcode": "string | null",
          "type": "emergency_location_primary | emergency_location_alternative | caller_location | transport_location  | unclear | other",
          "details": "string | null",
          "confidence": "number between 0 and 1"
        }
      ]
    }
    Erfasse dort alle Address-Kandidaten und versuche eine Confidence anzugeben, wie sicher du dir mit den Eingaben bist. Wenn du mehrere Kandidaten hast, dann nehme alle Kandidaten mit auf. Gebe nur einmal emergency_location_primary an bei der Adresse die du am wahrscheinlichsten für die entgültige Einsatzadresse siehst. Für weitere Alternativen nutze emergency_location_alternative Halluziniere nicht, erfinde nichts dazu, aber suche alle Informationen die du finden kannst.
    """

    user_prompt = f"""
    Das folgende Transcript soll auf Adressen untersucht werden: \n
    {dialogue}
    """

    response = client.responses.create(
    model="gpt-5.4",
    instructions=SYSTEM_PROMPT,
    input=user_prompt
    )

    #return response.output_text
    address_list = llm_to_addresses(parse_agent_response(response.output_text))

    return address_list

def normalize_text(s: str|None) -> str:
    if pd.isna(s):
        return ""
    s = str(s).strip().lower()
    s = (s.replace("ä","ae").replace("ö","oe").replace("ü","ue").replace("ß","ss"))
    s = re.sub(r"[.,]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    s = s.replace("straße", "strasse")
    s = re.sub(r"\bstr\.\b", "strasse", s)
    return s

def normalize_hn(hn: str|None) -> str:
    if pd.isna(hn):
        return ""
    return re.sub(r"\s+", "", str(hn).strip().upper())  # "12 a" -> "12A"

def fuzzy_best(value: str, candidates: list[str], min_score: int):
    m = process.extractOne(value, candidates, scorer=fuzz.token_sort_ratio)
    if m and m[1] >= min_score:
        return m[0], m[1]
    return None


def clean_value(value):
    """Wandelt pandas-/NumPy-Fehlwerte in None um."""
    if value is None:
        return None

    if isinstance(value, (list, dict, tuple, set)):
        return value

    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass

    return value


#Hinweis: Fuzzy-Logic gemeinsam mit ChatGPT entwickelt
def find_address_adaptive(
    address: Address,
    city_schedule=(92, 88, 85, 82, 78, 75, 72),
    street_schedule=(92, 90, 88, 85, 82, 80, 78, 75),
    min_rows=1,
    max_rows=25,
    require_same_city_for_street=True,
    min_street_len_for_fuzzy=4,
) -> list[Address]:

    c_in = normalize_text(address.city)
    s_in = normalize_text(address.street)
    hn_in = normalize_hn(address.housenumber)

    df = addresses
    work = df

    def to_addresses(
        matches: pd.DataFrame,
        fuzzy_mode: dict
    ) -> list[Address]:

        addresses = []



        for df_index, row in matches.head(max_rows).iterrows():
            candidate = Address(
                osm_id=clean_value(row.get("osm_id")),
                city=clean_value(row.get("city")),
                street=clean_value(row.get("street")),
                housenumber=clean_value(row.get("housenumber")),
                postcode=clean_value(row.get("postcode")),
                type=address.type,
                source=SourceEnum.ADDRESS_DATABASE,
                fuzzy_mode=fuzzy_mode.copy(),
            )

            addresses.append(candidate)

        return addresses

    # 0) Exakt
    exact = work.loc[
        (work["city_norm"] == c_in)
        & (work["street_norm"] == s_in)
        & (work["hn_norm"] == hn_in)
    ]

    if len(exact) >= min_rows:
        return to_addresses(
            exact,
            {
                "mode": "exact",
                "city_score": 100,
                "street_score": 100,
            }
        )

    # 1) Adaptive City-Fuzzy
    city_candidates = (
        work["city_norm"]
        .dropna()
        .unique()
        .tolist()
    )

    city_best = None
    city_score = None

    for cs in city_schedule:
        match = fuzzy_best(
            c_in,
            city_candidates,
            min_score=cs
        )

        if match:
            city_best, city_score = match
            break

    city_scope = (
        work
        if not city_best
        else work[work["city_norm"] == city_best]
    )

    # 2) Innerhalb der Stadt: exakte Straße und Hausnummer
    exact_street = city_scope.loc[
        (city_scope["street_norm"] == s_in)
        & (city_scope["hn_norm"] == hn_in)
    ]

    if len(exact_street) >= min_rows:
        return to_addresses(
            exact_street,
            {
                "mode": "city_exact_street",
                "city_score": city_score,
                "street_score": 100,
            }
        )

    # 3) Adaptive Street-Fuzzy innerhalb der Stadt
    street_candidates = (
        city_scope["street_norm"]
        .dropna()
        .unique()
        .tolist()
    )

    if len(s_in) >= min_street_len_for_fuzzy:
        for ss in street_schedule:
            match = fuzzy_best(
                s_in,
                street_candidates,
                min_score=ss
            )

            if match:
                street_best, street_score = match

                hits = city_scope.loc[
                    (city_scope["street_norm"] == street_best)
                    & (city_scope["hn_norm"] == hn_in)
                ]

                if len(hits) >= min_rows:
                    return to_addresses(
                        hits,
                        {
                            "mode": "city_fuzzy_street",
                            "city_score": city_score,
                            "street_score": street_score,
                        }
                    )

    # 4) Contains-Fallback
    seed = s_in[:max(3, len(s_in) // 2)]

    fallback = city_scope.loc[
        (city_scope["hn_norm"] == hn_in)
        & (
            city_scope["street_norm"].str.contains(
                re.escape(seed),
                na=False
            )
        )
    ]

    if len(fallback) > 0:
        return to_addresses(
            fallback,
            {
                "mode": "contains_fallback",
                "city_score": city_score,
                "street_score": None,
            }
        )

    # 5) Optional: globales Street-Fuzzy
    if (
        not require_same_city_for_street
        and len(s_in) >= min_street_len_for_fuzzy
    ):
        global_streets = (
            df["street_norm"]
            .dropna()
            .unique()
            .tolist()
        )

        for ss in street_schedule:
            match = fuzzy_best(
                s_in,
                global_streets,
                min_score=ss
            )

            if match:
                street_best, street_score = match

                hits = df.loc[
                    (df["street_norm"] == street_best)
                    & (df["hn_norm"] == hn_in)
                ]

                if len(hits) >= min_rows:
                    return to_addresses(
                        hits,
                        {
                            "mode": "global_fuzzy_street",
                            "city_score": None,
                            "street_score": street_score,
                        }
                    )

    return []

def flatten_addresses(address_list):
    result = []

    def flatten(item):

        if item is None:
            return
        if isinstance(item, (list, tuple)):
            for element in item:
                flatten(element)
        else:
            result.append(item)

    flatten(address_list)
    return result


def get_distances(address_list, aml_lat, aml_lon):
    #iterieren durch alle Adresskandidaten
    lat1 = aml_lat
    lon1 = aml_lon
    res = []
    for address in address_list:
        if address.type == AddressType.EMERGENCY_LOCATION_PRIMARY or address.type == AddressType.EMERGENCY_LOCATION_ALTERNATIVE:
            try:
                osm_id = address.osm_id
                lat2 = addresses.loc[addresses['osm_id']==osm_id,'lat'].iloc[0]
                address.lat = lat2
                lon2 = addresses.loc[addresses['osm_id']==osm_id,'lon'].iloc[0]
                address.lon = lon2
                address.gps_distance = haversine_km(lat1, lon1, lat2, lon2)
                # Aufrufen, zurückgeben, in res speichern, als emergency_addrss_candidates zurückgeben
                res.append(address)

            except Exception as e:
                address.gps_distance = None
                res.append(address)
                print(e)

    return res


def get_addresses_from_db(candidate_list, aml_lat=None, aml_lon=None):
    """
    Gibt eine flache Liste mit allen Adressen zurück, so wie sie in der DB gefunden werden.
    Der confidence-Wert aus den ursprünglichen Address-Candidates bleibt erhalten.
    """

    address_list = []

    for candidate in candidate_list:
        found_addresses = find_address_adaptive(candidate)

        for found_address in found_addresses:
            found_address = found_address.model_copy(
                update={
                    "confidence": candidate.confidence
                }
            )
            address_list.append(found_address)

    if aml_lat is not None and aml_lon is not None:
        address_list = get_distances(address_list, aml_lat, aml_lon)

    address_list = sorted(
        address_list,
        key=lambda a: float("inf") if a.gps_distance is None else a.gps_distance
    )

    return address_list

def on_scene_agent(dialogue):

    SYSTEM_PROMPT = """
    Du erhältst das Transkript eines deutschen Notrufgesprächs.

    Deine Aufgabe ist es, ausschließlich anhand des Transkripts zu bestimmen, ob sich die anrufende Person selbst am Notfallort befindet.

    Mögliche Fälle:

    * "yes": Die anrufende Person befindet sich am Notfallort.
    * "no": Die anrufende Person befindet sich nicht am Notfallort, beispielsweise weil sie für eine dritte Person anruft.
    * "unknown": Aus dem Transkript lässt sich nicht zuverlässig bestimmen, ob die anrufende Person am Notfallort ist.

    Beachte:

    * Das Gespräch wurde automatisch transkribiert und kann Transkriptionsfehler enthalten.
    * Berücksichtige daher den Gesprächskontext und mögliche phonetische Ähnlichkeiten.
    * Unterscheide zwischen der anrufenden Person und der betroffenen Person.
    * Ziehe keine Schlussfolgerungen, die nicht durch das Transkript gestützt werden.
    * Eine genannte Adresse allein beweist nicht, dass die anrufende Person vor Ort ist.
    * Aussagen wie „ich bin bei ihr“, „ich stehe daneben“ oder Beschreibungen aus unmittelbarer Wahrnehmung sprechen für "yes".
    * Aussagen wie „meine Mutter hat mich angerufen“, „ich bin nicht dort“ oder „ich rufe für meinen Nachbarn an“ sprechen für "no".
    * Wenn widersprüchliche oder keine relevanten Informationen vorliegen, verwende "unknown".

    Gib deine Antwort ausschließlich als valides JSON in folgendem Format zurück:

    {
    "findings": [

        response = client.responses.create(
        model="gpt-5.4",
        instructions=SYSTEM_PROMPT,
        input=user_prompt
        )
        "Wörtliches Zitat aus dem Transkript",
    "Weiteres relevantes wörtliches Zitat"
    ],
    "onscene_status": "yes",
    "confidence": 0.95
    }

    Regeln:

    * "findings" muss eine Liste mit ausschließlich wörtlichen Zitaten aus dem Transkript sein.
    * Nimm nur Zitate auf, die für die Bestimmung des Aufenthaltsorts der anrufenden Person relevant sind.
    * Wenn keine relevanten Aussagen vorhanden sind, gib eine leere Liste zurück.
    * "onscene_status" muss genau einen der Werte "yes", "no" oder "unknown" enthalten.
    * "confidence" muss eine Zahl zwischen 0.0 und 1.0 sein.
    * Die Konfidenz gibt an, wie sicher die gewählte Einstufung anhand des Transkripts ist.
    * Gib keinen zusätzlichen Text und keine Markdown-Codeblöcke aus.
      """

    user_prompt = f"""
    Das folgende Transcript soll untersucht werden: \n
    {dialogue}
    """

    response = client.responses.create(
    model="gpt-5.4",
    instructions=SYSTEM_PROMPT,
    input=user_prompt
    )

    out = parse_agent_response(response.output_text)
    return out.get('onscene_status'), out.get('confidence')

#Nächster Schritt: AML Agent: Adressen um die GPS-Position bekommen

def haversine_km(lat1, lon1, lat2, lon2):
    lat1 = math.radians(lat1)
    lon1 = math.radians(lon1)
    lat2 = math.radians(lat2)
    lon2 = math.radians(lon2)
    R = 6371.0088
    dlat, dlon = lat2 - lat1, lon2 - lon1
    h = math.sin(dlat/2)**2 + math.cos(lat1)*math.cos(lat2) * math.sin(dlon/2)**2
    return 2 * R * math.asin(math.sqrt(h))

# Adressen finden im Umkreis um die GPS-Koordinaten:
def find_addresses_within_radius(lat, lon, radius_km):
    lat_delta = radius_km / 111.0
    lon_delta = radius_km / (111.0 * math.cos(math.radians(lat)))

    #Erstmal Box drumrum finden
    candidates = addresses[
        (addresses["lat"] >= lat - lat_delta) &
        (addresses["lat"] <= lat + lat_delta) &
        (addresses["lon"] >= lon - lon_delta) &
        (addresses["lon"] <= lon + lon_delta)
    ].copy()

    #Exakte distanz nur für wenige Adressen
    candidates["distance_km"] = candidates.apply(
        lambda row: haversine_km(lat, lon, row["lat"], row["lon"]),
        axis=1
    )

    #Radius behandeln
    result = candidates[candidates["distance_km"] <= radius_km].sort_values("distance_km")

    return result

# 45, 75, 150, 750 = 1,5 fache Abstände

def get_addresses_radius(case, radius_factor = 1):

    aml_lat = case["aml_lat"]
    aml_lon = case["aml_lon"]
    aml_acc = case['aml_acc']
    if aml_acc == "high accuracy": radius = 0.03 * radius_factor
    elif aml_acc == "medium accuracy": radius = 0.05 * radius_factor
    elif aml_acc == "low accuracy": radius = 0.1 * radius_factor
    else: radius = 0.5 * radius_factor
    res = find_addresses_within_radius(
        aml_lat, aml_lon, radius
    )

    return res

# Ermittlung der Städte
def get_cities(nearby):
    return nearby["city"].value_counts(sort=False)

def get_streets(nearby, city=None):
    if city:
        df=nearby[nearby["city"] == city].copy()
        return df["street"].value_counts(sort=False)

    return nearby["street"].value_counts(sort=False)


# Suchfunktion von ChatGPT
def find_list_items_in_transcript(
    transcript: str,
    items: list[str],
    min_score: int = 80,
    exact_bonus: int = 100,
) -> list[dict]:
    """
    Prüft, ob Elemente aus einer einfachen String-Liste im Transkript vorkommen.

    Gibt pro Item zurück:
    - item: Originalwert
    - item_norm: normalisierte Form
    - found: True/False
    - score: Match-Score von 0 bis 100
    - match_type: exact, fuzzy oder none
    - matched_text: Textstelle bzw. bestes gefundenes Fragment
    """

    transcript_norm = normalize_text(transcript)

    results = []

    for item in items:
        item_norm = normalize_text(item)

        if not item_norm:
            results.append({
                "item": item,
                "item_norm": item_norm,
                "found": False,
                "score": 0,
                "match_type": "empty",
                "matched_text": None,
            })
            continue

        # 1) Exakter Treffer im normalisierten Transkript
        if item_norm in transcript_norm:
            results.append({
                "item": item,
                "item_norm": item_norm,
                "found": True,
                "score": exact_bonus,
                "match_type": "exact",
                "matched_text": item_norm,
            })
            continue

        # 2) Fuzzy-Treffer gegen Textfenster
        item_words = item_norm.split()
        transcript_words = transcript_norm.split()

        window_size = len(item_words)

        windows = [
            " ".join(transcript_words[i:i + window_size])
            for i in range(0, len(transcript_words) - window_size + 1)
        ]

        if not windows:
            results.append({
                "item": item,
                "item_norm": item_norm,
                "found": False,
                "score": 0,
                "match_type": "none",
                "matched_text": None,
            })
            continue

        match = process.extractOne(
            item_norm,
            windows,
            scorer=fuzz.token_sort_ratio
        )

        if match and match[1] >= min_score:
            matched_text, score, _ = match

            results.append({
                "item": item,
                "item_norm": item_norm,
                "found": True,
                "score": score,
                "match_type": "fuzzy",
                "matched_text": matched_text,
            })
        else:
            results.append({
                "item": item,
                "item_norm": item_norm,
                "found": False,
                "score": match[1] if match else 0,
                "match_type": "none",
                "matched_text": match[0] if match else None,
            })

    return results


def find_in_trancript(transcript, search_list):
    res = find_list_items_in_transcript(transcript, search_list)
    result = []
    for r in res:
        if r["found"]:
            result.append((r['item'], r['score']))

    return sorted(result, key=lambda x: x[1])


In [143]:
case = get_case()

In [144]:
# Plausible Adresses muss für jeden Einsatz neu erzeugt werden
# Vielleicht macht es sinn plausible Adresses erstmal leer zu lassen?
global addresses
addresses = pd.read_csv("../addresses/saarland_addresses_norm.csv")
global plausible_addresses
plausible_addresses = addresses.copy()
plausible_addresses['aml_score'] = 0
plausible_addresses['location_score'] = 0
plausible_addresses['address_agent_score'] = 0
plausible_addresses['address_score'] = 0
plausible_addresses['address_agent_confidence_score']= 0
plausible_addresses['db_fuzzy_score']= 0
plausible_addresses['address_agent_score'] = 0
plausible_addresses['address_agent_distance_score'] =0
plausible_addresses['address_score'] = 0
plausible_addresses

,Unnamed: 0,osm_id,street,housenumber,postcode,city,lat,lon,source,city_norm,street_norm,hn_norm,aml_score,location_score,address_agent_score,address_score,address_agent_confidence_score,db_fuzzy_score,address_agent_distance_score
0,0,29444682,Zweibrücker Straße,53,66459.0,Kirkel,49.307975,7.295332,addr-on-node,kirkel,zweibruecker strasse,53,0,0,0,0,0,0,0
1,1,50554264,Schloßberg-Höhen-Straße,1,66424.0,Homburg,49.320506,7.342649,addr-on-node,homburg,schlossberg-hoehen-strasse,1,0,0,0,0,0,0,0
2,2,78469844,Merziger Straße,85,66763.0,Dillingen/Saar,49.356546,6.720017,addr-on-node,dillingen/saar,merziger strasse,85,0,0,0,0,0,0,0
3,3,245597940,Rathausstraße,55,66333.0,Völklingen,49.248807,6.849234,addr-on-node,voelklingen,rathausstrasse,55,0,0,0,0,0,0,0
4,4,247314889,Großblittersdorfer Straße,285,66119.0,Saarbrücken,49.195252,7.023641,addr-on-node,saarbruecken,grossblittersdorfer strasse,285,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
332408,332408,1442432145,Anna-Simon-Weg,15,66822.0,Lebach,49.398598,6.918730,addr-on-way-centroid,lebach,anna-simon-weg,15,0,0,0,0,0,0,0
332409,332409,1442432147,Anna-Simon-Weg,21,66822.0,Lebach,49.399091,6.918981,addr-on-way-centroid,lebach,anna-simon-weg,21,0,0,0,0,0,0,0
332410,332410,1442432148,Anna-Simon-Weg,28,66822.0,Lebach,49.399029,6.919276,addr-on-way-centroid,lebach,anna-simon-weg,28,0,0,0,0,0,0,0
332411,332411,1442434884,Labachstraße,127,66793.0,Saarwellingen,49.357312,6.860812,addr-on-way-centroid,saarwellingen,labachstrasse,127,0,0,0,0,0,0,0


## AML-Score
Der AML Score kann direkt berechnet werden, sobald AML-Daten Vorliegen
Wenn keine Daten Vorliegen wird der Score zu 0

In [145]:
def get_aml_radius(acc):
    if acc == "high accuracy":
        return 0.030
    elif acc == "medium accuracy":
        return 0.050
    elif acc == "low accuracy":
        return 0.1
    else:
        return 0.5

def get_distance_score(distance, acc):
    # Der unterschiedlich große Radius bei den verschiedenen AML accuraties wird berücksichtigt
    acc_distance = get_aml_radius(acc)
    # Distance store gibt linear absteigende Werte für die Entfernung außerhalb des Radius bis zum 3 fachen radius
    if distance is None or acc_distance is None or acc_distance <= 0:
        return 0.0

    if distance <= acc_distance:
        return 1.0

    if distance >= 5 * acc_distance:
        return 0.0

    return (5 * acc_distance - distance) / (4 * acc_distance)


# Berücksichtig die aml_accuracy, da bei großer Accuracy die Daten grundsätzlich stärker berücksichtigt werden sollten -> Werte Müssen natürlich in der realität empirisch überprüft werden
def get_aml_correction_score(acc):
    if acc == "high accuracy":
        return 1
    elif acc == "medium accuracy":
        return 0.9
    elif acc == "low accuracy":
        return 0.7
    else:
        return 0.5

def get_aml_score(distance, acc):
    if acc is None or distance is None:
        return 0.0

    distance_score = get_distance_score(distance, acc)
    aml_correction_score = get_aml_correction_score(acc)

    return distance_score * aml_correction_score


def get_radius_addresses(case):
    candidates = get_addresses_radius(case, 5).copy()

    candidates["aml_score"] = candidates["distance_km"].apply(
        lambda d: get_aml_score(d, case["aml_acc"])
    )


    score_map = (
        candidates
        .groupby("osm_id")["aml_score"]
        .max()
    )

    plausible_addresses["aml_score"] = (
        plausible_addresses["osm_id"]
        .map(score_map)
        .fillna(0.0)
    )

    return candidates

# Plausible Adresses wird so angepasst, dass der aml_score übernommen wird



In [146]:

get_radius_addresses(case)

,Unnamed: 0,osm_id,street,housenumber,postcode,city,lat,lon,source,city_norm,street_norm,hn_norm,distance_km,aml_score
620,620,1569809880,Berliner Straße,123,66424.0,Homburg,49.340189,7.336976,addr-on-node,homburg,berliner strasse,123,0.010091,0.900000
621,621,1569809888,Berliner Straße,104-106,66424.0,NaN,49.340019,7.336603,addr-on-node,NaN,berliner strasse,104-106,0.036694,0.900000
245143,245143,360305160,Berliner Straße,104-106,66424.0,Homburg,49.339781,7.336296,addr-on-way-centroid,homburg,berliner strasse,104-106,0.071287,0.804207
45698,45698,143018147,Berliner Straße,125,66424.0,Homburg,49.340100,7.337974,addr-on-way-centroid,homburg,berliner strasse,125,0.078352,0.772418
45697,45697,142926162,Berliner Straße,120,66424.0,Homburg,49.339496,7.337103,addr-on-way-centroid,homburg,berliner strasse,120,0.087338,0.731977
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
244875,244875,360304755,Dürerstraße,134,66424.0,Homburg,49.338713,7.334575,addr-on-way-centroid,homburg,duererstrasse,134,0.243167,0.030747
257603,257603,370555181,Baldungstraße,6,66424.0,Homburg,49.342150,7.338671,addr-on-way-centroid,homburg,baldungstrasse,6,0.244028,0.026875
307145,307145,707356573,Rubensstraße,4,66424.0,Homburg,49.341779,7.334420,addr-on-way-centroid,homburg,rubensstrasse,4,0.246905,0.013925
301503,301503,674275793,Rubensstraße,5,66424.0,Homburg,49.341583,7.334153,addr-on-way-centroid,homburg,rubensstrasse,5,0.248095,0.008573


In [147]:
plausible_addresses.sort_values(
    by="aml_score",
    ascending=False
)

,Unnamed: 0,osm_id,street,housenumber,postcode,city,lat,lon,source,city_norm,street_norm,hn_norm,aml_score,location_score,address_agent_score,address_score,address_agent_confidence_score,db_fuzzy_score,address_agent_distance_score
620,620,1569809880,Berliner Straße,123,66424.0,Homburg,49.340189,7.336976,addr-on-node,homburg,berliner strasse,123,0.900000,0,0,0,0,0,0
621,621,1569809888,Berliner Straße,104-106,66424.0,NaN,49.340019,7.336603,addr-on-node,NaN,berliner strasse,104-106,0.900000,0,0,0,0,0,0
245143,245143,360305160,Berliner Straße,104-106,66424.0,Homburg,49.339781,7.336296,addr-on-way-centroid,homburg,berliner strasse,104-106,0.804207,0,0,0,0,0,0
45698,45698,143018147,Berliner Straße,125,66424.0,Homburg,49.340100,7.337974,addr-on-way-centroid,homburg,berliner strasse,125,0.772418,0,0,0,0,0,0
45697,45697,142926162,Berliner Straße,120,66424.0,Homburg,49.339496,7.337103,addr-on-way-centroid,homburg,berliner strasse,120,0.731977,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110807,110807,257782972,Lindenstraße,7,66839.0,Schmelz,49.440152,6.857095,addr-on-way-centroid,schmelz,lindenstrasse,7,0.000000,0,0,0,0,0,0
110806,110806,257782971,Lindenstraße,1a,66839.0,Schmelz,49.440471,6.857006,addr-on-way-centroid,schmelz,lindenstrasse,1A,0.000000,0,0,0,0,0,0
110805,110805,257782969,Lindenstraße,3,66839.0,Schmelz,49.440340,6.857034,addr-on-way-centroid,schmelz,lindenstrasse,3,0.000000,0,0,0,0,0,0
110804,110804,257782967,Lindenstraße,11,66839.0,Schmelz,49.440025,6.857134,addr-on-way-centroid,schmelz,lindenstrasse,11,0.000000,0,0,0,0,0,0


## OnSceneScore
Der OnSceneScore ist zunächst abhängig von der KI Einstufung, kann aber vom human überschrieben werden.
Defailt Wert ist 0.5

## Location Score
on_scene_score * aml_score

In [148]:
def get_on_scene_score(on_scene_agent_result, on_scene_agent_conf, on_scene_human_result = None):
    if on_scene_human_result is None:
        if on_scene_agent_result == 'yes':
            return on_scene_agent_conf
        elif on_scene_agent_result == 'no':
            return 1 - on_scene_agent_conf
    elif on_scene_human_result:
        return 1
    elif on_scene_human_result == False:
        return 0

    return 0.5

def add_location_score(case, on_scene_human_result = None):
    on_scene_agent_result, get_on_scene_conf = 'unknown', 0
    if on_scene_human_result is None:
        on_scene_agent_result, get_on_scene_conf = on_scene_agent(case['dialogue_transcripted'])
    on_scene_score = get_on_scene_score(on_scene_agent_result, get_on_scene_conf, on_scene_human_result)
    plausible_addresses['location_score'] = plausible_addresses['aml_score'] * on_scene_score
    return on_scene_score

In [149]:


add_location_score(case)

0.010000000000000009

In [150]:
plausible_addresses.sort_values(
    by="location_score",
    ascending=False
)

,Unnamed: 0,osm_id,street,housenumber,postcode,city,lat,lon,source,city_norm,street_norm,hn_norm,aml_score,location_score,address_agent_score,address_score,address_agent_confidence_score,db_fuzzy_score,address_agent_distance_score
620,620,1569809880,Berliner Straße,123,66424.0,Homburg,49.340189,7.336976,addr-on-node,homburg,berliner strasse,123,0.900000,0.009000,0,0,0,0,0
621,621,1569809888,Berliner Straße,104-106,66424.0,NaN,49.340019,7.336603,addr-on-node,NaN,berliner strasse,104-106,0.900000,0.009000,0,0,0,0,0
245143,245143,360305160,Berliner Straße,104-106,66424.0,Homburg,49.339781,7.336296,addr-on-way-centroid,homburg,berliner strasse,104-106,0.804207,0.008042,0,0,0,0,0
45698,45698,143018147,Berliner Straße,125,66424.0,Homburg,49.340100,7.337974,addr-on-way-centroid,homburg,berliner strasse,125,0.772418,0.007724,0,0,0,0,0
45697,45697,142926162,Berliner Straße,120,66424.0,Homburg,49.339496,7.337103,addr-on-way-centroid,homburg,berliner strasse,120,0.731977,0.007320,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110807,110807,257782972,Lindenstraße,7,66839.0,Schmelz,49.440152,6.857095,addr-on-way-centroid,schmelz,lindenstrasse,7,0.000000,0.000000,0,0,0,0,0
110806,110806,257782971,Lindenstraße,1a,66839.0,Schmelz,49.440471,6.857006,addr-on-way-centroid,schmelz,lindenstrasse,1A,0.000000,0.000000,0,0,0,0,0
110805,110805,257782969,Lindenstraße,3,66839.0,Schmelz,49.440340,6.857034,addr-on-way-centroid,schmelz,lindenstrasse,3,0.000000,0.000000,0,0,0,0,0
110804,110804,257782967,Lindenstraße,11,66839.0,Schmelz,49.440025,6.857134,addr-on-way-centroid,schmelz,lindenstrasse,11,0.000000,0.000000,0,0,0,0,0


## Address Agent Score

In [151]:
def get_address_agent_score(case):

    address_list = address_agent(case["dialogue"])
    db_list = get_addresses_from_db(address_list)

    score_cols = [
        "address_agent_confidence_score",
        "db_fuzzy_score",
        "address_agent_score"
    ]

    for col in score_cols:
        if col not in plausible_addresses.columns:
            plausible_addresses[col] = np.nan
        else:
            plausible_addresses[col] = plausible_addresses[col].astype(float)

    for add in db_list:
        confidence = add.confidence if add.confidence is not None else 0.0

        fuzzy_mode = add.fuzzy_mode or {}

        city_score = fuzzy_mode.get("city_score", 0) / 100
        street_score = fuzzy_mode.get("street_score", 0) / 100

        db_fuzzy_score = city_score * street_score
        address_agent_score = confidence * db_fuzzy_score

        mask = plausible_addresses["osm_id"] == add.osm_id

        plausible_addresses.loc[mask, "address_agent_confidence_score"] = confidence
        plausible_addresses.loc[mask, "db_fuzzy_score"] = db_fuzzy_score
        plausible_addresses.loc[mask, "address_agent_score"] = address_agent_score

    return

In [152]:


get_address_agent_score(case)

In [153]:
plausible_addresses.sort_values(
    by="address_agent_confidence_score",
    ascending=False
)

,Unnamed: 0,osm_id,street,housenumber,postcode,city,lat,lon,source,city_norm,street_norm,hn_norm,aml_score,location_score,address_agent_score,address_score,address_agent_confidence_score,db_fuzzy_score,address_agent_distance_score
290777,290777,589343958,Gartenweg,18,66589.0,Merchweiler,49.356190,7.057493,addr-on-way-centroid,merchweiler,gartenweg,18,0.0,0.0,0.99,0,0.99,1.0,0
0,0,29444682,Zweibrücker Straße,53,66459.0,Kirkel,49.307975,7.295332,addr-on-node,kirkel,zweibruecker strasse,53,0.0,0.0,0.00,0,0.00,0.0,0
221605,221605,353811747,Saarwellinger Straße,88,66740.0,Saarlouis,49.334437,6.756871,addr-on-way-centroid,saarlouis,saarwellinger strasse,88,0.0,0.0,0.00,0,0.00,0.0,0
221612,221612,353811756,Rodenhübel,31,66740.0,Saarlouis,49.336302,6.754357,addr-on-way-centroid,saarlouis,rodenhuebel,31,0.0,0.0,0.00,0,0.00,0.0,0
221611,221611,353811754,Werderstraße,136,66763.0,Dillingen/Saar,49.360238,6.735027,addr-on-way-centroid,dillingen/saar,werderstrasse,136,0.0,0.0,0.00,0,0.00,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110803,110803,257782964,Lindenstraße,13,66839.0,Schmelz,49.439964,6.857163,addr-on-way-centroid,schmelz,lindenstrasse,13,0.0,0.0,0.00,0,0.00,0.0,0
110802,110802,257781789,Lindenstraße,44,66839.0,Schmelz,49.437307,6.857410,addr-on-way-centroid,schmelz,lindenstrasse,44,0.0,0.0,0.00,0,0.00,0.0,0
110801,110801,257781788,Fichtenstraße,54,66839.0,Schmelz,49.430716,6.857644,addr-on-way-centroid,schmelz,fichtenstrasse,54,0.0,0.0,0.00,0,0.00,0.0,0
110800,110800,257781787,Fichtenstraße,49,66839.0,Schmelz,49.430699,6.858041,addr-on-way-centroid,schmelz,fichtenstrasse,49,0.0,0.0,0.00,0,0.00,0.0,0


In [154]:
def get_address_agent_distance_score(row):
    return row['location_score'] * row['address_agent_score']

def get_address_score(row):
    return row['address_agent_distance_score'] + row['address_agent_score'] + row['location_score']

plausible_addresses['address_agent_distance_score'] = plausible_addresses.apply(get_address_agent_distance_score, axis=1)
plausible_addresses['address_score'] = plausible_addresses.apply(get_address_score, axis=1)


In [155]:
plausible_addresses.sort_values(
    by="address_score",
    ascending=False
)

,Unnamed: 0,osm_id,street,housenumber,postcode,city,lat,lon,source,city_norm,street_norm,hn_norm,aml_score,location_score,address_agent_score,address_score,address_agent_confidence_score,db_fuzzy_score,address_agent_distance_score
290777,290777,589343958,Gartenweg,18,66589.0,Merchweiler,49.356190,7.057493,addr-on-way-centroid,merchweiler,gartenweg,18,0.0,0.0,0.99,0.99,0.99,1.0,0.0
0,0,29444682,Zweibrücker Straße,53,66459.0,Kirkel,49.307975,7.295332,addr-on-node,kirkel,zweibruecker strasse,53,0.0,0.0,0.00,0.00,0.00,0.0,0.0
221605,221605,353811747,Saarwellinger Straße,88,66740.0,Saarlouis,49.334437,6.756871,addr-on-way-centroid,saarlouis,saarwellinger strasse,88,0.0,0.0,0.00,0.00,0.00,0.0,0.0
221612,221612,353811756,Rodenhübel,31,66740.0,Saarlouis,49.336302,6.754357,addr-on-way-centroid,saarlouis,rodenhuebel,31,0.0,0.0,0.00,0.00,0.00,0.0,0.0
221611,221611,353811754,Werderstraße,136,66763.0,Dillingen/Saar,49.360238,6.735027,addr-on-way-centroid,dillingen/saar,werderstrasse,136,0.0,0.0,0.00,0.00,0.00,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110803,110803,257782964,Lindenstraße,13,66839.0,Schmelz,49.439964,6.857163,addr-on-way-centroid,schmelz,lindenstrasse,13,0.0,0.0,0.00,0.00,0.00,0.0,0.0
110802,110802,257781789,Lindenstraße,44,66839.0,Schmelz,49.437307,6.857410,addr-on-way-centroid,schmelz,lindenstrasse,44,0.0,0.0,0.00,0.00,0.00,0.0,0.0
110801,110801,257781788,Fichtenstraße,54,66839.0,Schmelz,49.430716,6.857644,addr-on-way-centroid,schmelz,fichtenstrasse,54,0.0,0.0,0.00,0.00,0.00,0.0,0.0
110800,110800,257781787,Fichtenstraße,49,66839.0,Schmelz,49.430699,6.858041,addr-on-way-centroid,schmelz,fichtenstrasse,49,0.0,0.0,0.00,0.00,0.00,0.0,0.0


In [164]:
case = get_case()
case

{'gold_address': Address(osm_id=323348005, city='Überherrn', postcode=66802, street='Lerchenweg', housenumber='5', lat=49.27343478721882, lon=6.70083792802993, gps_distance=None, type=<AddressType.GOLD: 'gold'>, special_notes=None, confidence=None, source=<SourceEnum.ADDRESS_DATABASE: 'address_database'>, fuzzy_mode=None),
 'onscene': True,
 'dialogue': {'dialogue': [{'speaker': 'calltaker',
    'text': 'Notruf Feuerwehr und Rettungsdienst, in welchem Ort ist der Notfall?'},
   {'speaker': 'caller',
    'text': 'Bitte kommen Sie schnell, es ist ganz schlimm hier!'},
   {'speaker': 'calltaker',
    'text': 'Ich schicke Hilfe. Sagen Sie mir bitte zuerst den Ort.'},
   {'speaker': 'caller',
    'text': 'Was? Hallo? Ich versteh Sie kaum! Überherrn, in Überherrn!'},
   {'speaker': 'calltaker',
    'text': 'Verstanden, in Überherrn. Wie lautet die Straße und Hausnummer?'},
   {'speaker': 'caller',
    'text': 'Im Lerchenweg, Lerchenweg 5! Bitte beeilen Sie sich!'},
   {'speaker': 'calltaker'

In [165]:
# Das ist noch einmal die gesamte Logik, die sich hier im Beispiel noch nacheinander abspielt, im Backend aber parallel laufen soll und sich bei jedem Schritt nochmal anpassen soll, das heißt wenn sich zum Beispiel onscene_score ändert, dann müssen sich natürlich auch alle Werte dynamisch ändern

plausible_addresses = addresses.copy()
plausible_addresses['aml_score'] = 0
plausible_addresses['location_score'] = 0
plausible_addresses['address_agent_score'] = 0
plausible_addresses['address_score'] = 0
plausible_addresses['address_agent_confidence_score']= 0
plausible_addresses['db_fuzzy_score']= 0
plausible_addresses['address_agent_score'] = 0
plausible_addresses['address_agent_distance_score'] =0
plausible_addresses['address_score'] = 0


get_radius_addresses(case)

add_location_score(case)

get_address_agent_score(case)
plausible_addresses['address_agent_distance_score'] = plausible_addresses.apply(get_address_agent_distance_score, axis=1)
plausible_addresses['address_score'] = plausible_addresses.apply(get_address_score, axis=1)

plausible_addresses.sort_values(
    by="address_score",
    ascending=False
)

,Unnamed: 0,osm_id,street,housenumber,postcode,city,lat,lon,source,city_norm,street_norm,hn_norm,aml_score,location_score,address_agent_score,address_score,address_agent_confidence_score,db_fuzzy_score,address_agent_distance_score
150209,150209,323348005,Lerchenweg,5,66802.0,Überherrn,49.273435,6.700838,addr-on-way-centroid,ueberherrn,lerchenweg,5,0.5,0.465,0.99,1.45035,0.99,1.0,0.46035
221615,221615,353811759,Saarwellinger Straße,68,66740.0,Saarlouis,49.333793,6.755842,addr-on-way-centroid,saarlouis,saarwellinger strasse,68,0.0,0.000,0.00,0.00000,0.00,0.0,0.00000
221613,221613,353811757,Werderstraße,118b,66763.0,Dillingen/Saar,49.359883,6.733543,addr-on-way-centroid,dillingen/saar,werderstrasse,118B,0.0,0.000,0.00,0.00000,0.00,0.0,0.00000
221612,221612,353811756,Rodenhübel,31,66740.0,Saarlouis,49.336302,6.754357,addr-on-way-centroid,saarlouis,rodenhuebel,31,0.0,0.000,0.00,0.00000,0.00,0.0,0.00000
221611,221611,353811754,Werderstraße,136,66763.0,Dillingen/Saar,49.360238,6.735027,addr-on-way-centroid,dillingen/saar,werderstrasse,136,0.0,0.000,0.00,0.00000,0.00,0.0,0.00000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110802,110802,257781789,Lindenstraße,44,66839.0,Schmelz,49.437307,6.857410,addr-on-way-centroid,schmelz,lindenstrasse,44,0.0,0.000,0.00,0.00000,0.00,0.0,0.00000
110801,110801,257781788,Fichtenstraße,54,66839.0,Schmelz,49.430716,6.857644,addr-on-way-centroid,schmelz,fichtenstrasse,54,0.0,0.000,0.00,0.00000,0.00,0.0,0.00000
110800,110800,257781787,Fichtenstraße,49,66839.0,Schmelz,49.430699,6.858041,addr-on-way-centroid,schmelz,fichtenstrasse,49,0.0,0.000,0.00,0.00000,0.00,0.0,0.00000
110799,110799,257781786,Fichtenstraße,45,66839.0,Schmelz,49.430952,6.858333,addr-on-way-centroid,schmelz,fichtenstrasse,45,0.0,0.000,0.00,0.00000,0.00,0.0,0.00000


In [ ]:
# Der User/ Calltaker soll ein paar möglichkeiten haben:
# Es muss ein Adressfeld bestehend aus Stadt, Straße, Hausnummer geben, in die der User eintippen kann.
# Wenn der KI-Dienst zu vorschlägen kommt, sollen die als Beispiel angezeigt werden

# Dauerhaft checken, ob Addresskandidaten besonders hervorstechen:
def get_candidates():
    df = plausible_addresses[plausible_addresses['address_score'] > 0]
    if len(df) > 0:
        #Zeige die Adresse als Vorschlag im Adressfeld
        cities = get_cities(df)
        #Soll natürlich nicht geprintet werden sondern im Frontend als Möglichkeit angezeigt werden
        print(cities[0:3])
        city = None
        # Wenn der user jetzt eine City auswählt oder Eintippt dann gilt:
        #city = Usereingabe
        streets = get_streets(df, city)
        #Soll natürlich nicht geprintet werden sondern im Frontend als Möglichkeit angezeigt werden
        print(streets[0:3])

# get candidates soll nach jedem run laufen, also immer wenn sich was verändert hat, so dass der User immer sieht was gerade die aktuell besten Kandiaten sind

# Auch soll der User die Möglichkeit haben, die einzelen Scores für einen Adress Kandidaten in einem Schaubild zu sehen. Bereite das vor, so dass wir das später im Frontend umsetzen können.
